# Pharmacy Inventory & Medicine Consumption Prediction
**Dataset:** Pharma Sales Data by Milan Zdravković
https://www.kaggle.com/datasets/milanzdravkovic/pharma-sales-data

**Drug Categories (ATC):**
- M01AB — Anti-inflammatory, Acetic acid derivatives (e.g. Diclofenac)
- M01AE — Anti-inflammatory, Propionic acid derivatives (e.g. Ibuprofen)
- N02BA — Analgesics, Salicylic acid derivatives (e.g. Aspirin)
- N02BE — Analgesics, Anilides (e.g. Paracetamol)
- N05B  — Anxiolytics (e.g. Diazepam)
- N05C  — Hypnotics/Sedatives
- R03   — Asthma/COPD drugs (e.g. Salbutamol)
- R06   — Antihistamines (e.g. Cetirizine)

**Goals:** Predict demand, classify stock status, generate reorder alerts

In [2]:
# pip install xgboost scikit-learn pandas numpy
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, classification_report
)
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

print('All imports successful.')

All imports successful.


In [3]:
# Download salesdaily.csv from Kaggle:
# https://www.kaggle.com/datasets/milanzdravkovic/pharma-sales-data
# Place it in the same folder as this notebook

df_raw = pd.read_csv('Data/salesdaily.csv')

print('Shape:', df_raw.shape)
print('Columns:', df_raw.columns.tolist())
print(df_raw.head())
print(df_raw.dtypes)

Shape: (2106, 13)
Columns: ['datum', 'M01AB', 'M01AE', 'N02BA', 'N02BE', 'N05B', 'N05C', 'R03', 'R06', 'Year', 'Month', 'Hour', 'Weekday Name']
      datum  M01AB  M01AE  N02BA  N02BE  N05B  N05C   R03  R06  Year  Month  \
0  1/2/2014    0.0   3.67    3.4  32.40   7.0   0.0   0.0  2.0  2014      1   
1  1/3/2014    8.0   4.00    4.4  50.60  16.0   0.0  20.0  4.0  2014      1   
2  1/4/2014    2.0   1.00    6.5  61.85  10.0   0.0   9.0  1.0  2014      1   
3  1/5/2014    4.0   3.00    7.0  41.10   8.0   0.0   3.0  0.0  2014      1   
4  1/6/2014    5.0   1.00    4.5  21.70  16.0   2.0   6.0  2.0  2014      1   

   Hour Weekday Name  
0   248     Thursday  
1   276       Friday  
2   276     Saturday  
3   276       Sunday  
4   276       Monday  
datum            object
M01AB           float64
M01AE           float64
N02BA           float64
N02BE           float64
N05B            float64
N05C            float64
R03             float64
R06             float64
Year              int64
Mon

In [4]:
# Parse date
df_raw['datum'] = pd.to_datetime(df_raw['datum'])
df_raw.rename(columns={'datum': 'date'}, inplace=True)
df_raw.sort_values('date', inplace=True)
df_raw.reset_index(drop=True, inplace=True)

drug_cols = ['M01AB', 'M01AE', 'N02BA', 'N02BE', 'N05B', 'N05C', 'R03', 'R06']

# Drug metadata
drug_meta = {
    'M01AB': {'category': 'Anti-inflammatory', 'lead_time_days': 3,  'unit_price': 20},
    'M01AE': {'category': 'Anti-inflammatory', 'lead_time_days': 3,  'unit_price': 15},
    'N02BA': {'category': 'Painkiller',         'lead_time_days': 3,  'unit_price': 10},
    'N02BE': {'category': 'Painkiller',         'lead_time_days': 3,  'unit_price': 5},
    'N05B':  {'category': 'Anxiolytic',          'lead_time_days': 7,  'unit_price': 50},
    'N05C':  {'category': 'Sedative',            'lead_time_days': 7,  'unit_price': 45},
    'R03':   {'category': 'Respiratory',         'lead_time_days': 10, 'unit_price': 120},
    'R06':   {'category': 'Antihistamine',       'lead_time_days': 5,  'unit_price': 25},
}

# Melt wide to long format
df = df_raw.melt(
    id_vars=['date'],
    value_vars=drug_cols,
    var_name='drug_name',
    value_name='quantity_dispensed'
)

df['category']       = df['drug_name'].map(lambda x: drug_meta[x]['category'])
df['lead_time_days'] = df['drug_name'].map(lambda x: drug_meta[x]['lead_time_days'])
df['unit_price']     = df['drug_name'].map(lambda x: drug_meta[x]['unit_price'])

df.sort_values(['drug_name', 'date'], inplace=True)
df.reset_index(drop=True, inplace=True)

print('Reshaped shape:', df.shape)
print('Date range:', df['date'].min(), 'to', df['date'].max())
print(df.groupby('drug_name')['quantity_dispensed'].describe().round(2))

Reshaped shape: (16848, 6)
Date range: 2014-01-02 00:00:00 to 2019-10-08 00:00:00
            count   mean    std  min    25%    50%    75%     max
drug_name                                                        
M01AB      2106.0   5.03   2.74  0.0   3.00   4.99   6.67   17.34
M01AE      2106.0   3.90   2.13  0.0   2.34   3.67   5.14   14.46
N02BA      2106.0   3.88   2.38  0.0   2.00   3.50   5.20   16.00
N02BE      2106.0  29.92  15.59  0.0  19.00  26.90  38.30  161.00
N05B       2106.0   8.85   5.61  0.0   5.00   8.00  12.00   54.83
N05C       2106.0   0.59   1.09  0.0   0.00   0.00   1.00    9.00
R03        2106.0   5.51   6.43  0.0   1.00   4.00   8.00   45.00
R06        2106.0   2.90   2.42  0.0   1.00   2.00   4.00   15.00


In [5]:
# Simulate stock levels (dataset only has sales, not stock)
# Assumes pharmacy restocks to 30-day supply when below 7-day supply

def simulate_stock(group):
    avg_daily = group['quantity_dispensed'].mean()
    stock = avg_daily * 30
    stocks = []
    for qty in group['quantity_dispensed']:
        stocks.append(round(stock))
        stock = max(0, stock - qty)
        if stock < avg_daily * 7:
            stock += avg_daily * 30
    group = group.copy()
    group['stock_level'] = stocks
    return group

df = df.groupby('drug_name', group_keys=False).apply(simulate_stock)
df.reset_index(drop=True, inplace=True)

print('Stock simulated.')
print(df[['drug_name','date','quantity_dispensed','stock_level']].head(10))

Stock simulated.
  drug_name       date  quantity_dispensed  stock_level
0     M01AB 2014-01-02                0.00          151
1     M01AB 2014-01-03                8.00          151
2     M01AB 2014-01-04                2.00          143
3     M01AB 2014-01-05                4.00          141
4     M01AB 2014-01-06                5.00          137
5     M01AB 2014-01-07                0.00          132
6     M01AB 2014-01-08                5.33          132
7     M01AB 2014-01-09                7.00          127
8     M01AB 2014-01-10                5.00          120
9     M01AB 2014-01-11                5.00          115


In [6]:
df_feat = df.copy()

# Date features
df_feat['day_of_week']    = df_feat['date'].dt.dayofweek
df_feat['day_of_month']   = df_feat['date'].dt.day
df_feat['month']          = df_feat['date'].dt.month
df_feat['year']           = df_feat['date'].dt.year
df_feat['week_of_year']   = df_feat['date'].dt.isocalendar().week.astype(int)
df_feat['quarter']        = df_feat['date'].dt.quarter
df_feat['is_weekend']     = (df_feat['day_of_week'] >= 5).astype(int)
df_feat['is_month_start'] = df_feat['date'].dt.is_month_start.astype(int)
df_feat['is_month_end']   = df_feat['date'].dt.is_month_end.astype(int)
df_feat['season']         = df_feat['month'].map(
    {12:1,1:1,2:1, 3:2,4:2,5:2, 6:3,7:3,8:3, 9:4,10:4,11:4}
)

# Rolling demand features (shift(1) avoids data leakage)
grp = df_feat.groupby('drug_name')['quantity_dispensed']
df_feat['demand_lag1']   = grp.shift(1)
df_feat['demand_lag7']   = grp.shift(7)
df_feat['demand_lag14']  = grp.shift(14)
df_feat['demand_lag30']  = grp.shift(30)

shifted = grp.shift(1)
df_feat['demand_roll7']  = shifted.groupby(df_feat['drug_name']).transform(lambda x: x.rolling(7,  min_periods=1).mean())
df_feat['demand_roll14'] = shifted.groupby(df_feat['drug_name']).transform(lambda x: x.rolling(14, min_periods=1).mean())
df_feat['demand_roll30'] = shifted.groupby(df_feat['drug_name']).transform(lambda x: x.rolling(30, min_periods=1).mean())
df_feat['demand_roll90'] = shifted.groupby(df_feat['drug_name']).transform(lambda x: x.rolling(90, min_periods=1).mean())
df_feat['demand_std7']   = shifted.groupby(df_feat['drug_name']).transform(lambda x: x.rolling(7,  min_periods=2).std())
df_feat['demand_std30']  = shifted.groupby(df_feat['drug_name']).transform(lambda x: x.rolling(30, min_periods=2).std())
df_feat['demand_trend']  = df_feat['demand_roll7'] - df_feat['demand_roll30']

# Stock features
df_feat['stock_lag1']    = df_feat.groupby('drug_name')['stock_level'].shift(1)
df_feat['days_of_stock'] = np.where(
    df_feat['demand_roll7'] > 0,
    df_feat['stock_level'] / df_feat['demand_roll7'],
    999
).clip(0, 365)

# Reorder point and safety stock
df_feat['safety_stock']  = (
    1.5 * df_feat['demand_std30'].fillna(df_feat['demand_roll30'] * 0.1)
    * np.sqrt(df_feat['lead_time_days'])
)
df_feat['reorder_point'] = (
    df_feat['demand_roll30'] * df_feat['lead_time_days'] + df_feat['safety_stock']
)

# Stock status labels
def assign_stock_status(row):
    if row['stock_level'] <= 0:
        return 'OUT_OF_STOCK'
    elif row['stock_level'] <= row['reorder_point']:
        return 'CRITICAL'
    elif row['stock_level'] <= row['reorder_point'] * 1.5:
        return 'LOW'
    elif row['stock_level'] >= row['demand_roll30'] * 90:
        return 'OVERSTOCK'
    else:
        return 'NORMAL'

df_feat['stock_status'] = df_feat.apply(assign_stock_status, axis=1)

# Encode categoricals
le_drug = LabelEncoder()
le_cat  = LabelEncoder()
df_feat['drug_enc']     = le_drug.fit_transform(df_feat['drug_name'])
df_feat['category_enc'] = le_cat.fit_transform(df_feat['category'])

df_feat.dropna(subset=['demand_lag30','demand_roll30','demand_std30'], inplace=True)
df_feat.reset_index(drop=True, inplace=True)

print('Feature engineering done. Shape:', df_feat.shape)
print('Stock status distribution:')
print(df_feat['stock_status'].value_counts())

Feature engineering done. Shape: (16608, 35)
Stock status distribution:
stock_status
NORMAL       13681
LOW           1635
CRITICAL      1183
OVERSTOCK      109
Name: count, dtype: int64


In [7]:
regression_features = [
    'drug_enc', 'category_enc',
    'day_of_week', 'day_of_month', 'month', 'year',
    'week_of_year', 'quarter', 'season',
    'is_weekend', 'is_month_start', 'is_month_end',
    'demand_lag1', 'demand_lag7', 'demand_lag14', 'demand_lag30',
    'demand_roll7', 'demand_roll14', 'demand_roll30', 'demand_roll90',
    'demand_std7', 'demand_std30', 'demand_trend',
    'stock_level', 'stock_lag1',
    'lead_time_days', 'unit_price',
]

classification_features = regression_features + [
    'days_of_stock', 'reorder_point', 'safety_stock'
]

print('Regression features:', len(regression_features))
print('Classification features:', len(classification_features))

Regression features: 27
Classification features: 30


In [8]:
# MODEL A: Demand Forecasting (time-based split — do NOT shuffle!)
X_reg = df_feat[regression_features].astype('float64')
y_reg = df_feat['quantity_dispensed'].astype('float64')

split_idx = int(len(df_feat) * 0.80)
X_train_r, X_test_r = X_reg.iloc[:split_idx], X_reg.iloc[split_idx:]
y_train_r, y_test_r = y_reg.iloc[:split_idx], y_reg.iloc[split_idx:]

print(f'Train: {X_train_r.shape} | Test: {X_test_r.shape}')

xgb_reg = xgb.XGBRegressor(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    min_child_weight=3, random_state=42, n_jobs=-1
)
xgb_reg.fit(X_train_r, y_train_r, eval_set=[(X_test_r, y_test_r)], verbose=False)

y_pred_r = np.maximum(xgb_reg.predict(X_test_r), 0)

print(f'\n── Demand Forecasting Results ──────────────────')
print(f'  MAE  : {mean_absolute_error(y_test_r, y_pred_r):.4f} units')
print(f'  RMSE : {np.sqrt(mean_squared_error(y_test_r, y_pred_r)):.4f} units')
print(f'  R2   : {r2_score(y_test_r, y_pred_r):.4f}')
print(f'  MAPE : {np.mean(np.abs((y_test_r - y_pred_r)/(y_test_r+1e-5)))*100:.2f}%')

Train: (13286, 27) | Test: (3322, 27)

── Demand Forecasting Results ──────────────────
  MAE  : 2.8759 units
  RMSE : 4.4333 units
  R2   : 0.1701
  MAPE : 5651172.93%


In [9]:
# MODEL B: Stock Status Classification
le_status = LabelEncoder()
y_cls = le_status.fit_transform(df_feat['stock_status'])
X_cls = df_feat[classification_features].astype('float64')

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cls, y_cls, test_size=0.20, random_state=42, stratify=y_cls
)

xgb_clf = xgb.XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    use_label_encoder=False, eval_metric='mlogloss',
    random_state=42, n_jobs=-1
)
xgb_clf.fit(X_train_c, y_train_c)

y_pred_c = xgb_clf.predict(X_test_c)
print('── Stock Status Classification ─────────────────────')
print(f'Accuracy: {accuracy_score(y_test_c, y_pred_c):.4f}')
print(classification_report(y_test_c, y_pred_c, target_names=le_status.classes_))

── Stock Status Classification ─────────────────────
Accuracy: 0.9828
              precision    recall  f1-score   support

    CRITICAL       0.99      0.95      0.97       237
         LOW       0.91      0.93      0.92       327
      NORMAL       0.99      0.99      0.99      2736
   OVERSTOCK       0.95      0.86      0.90        22

    accuracy                           0.98      3322
   macro avg       0.96      0.93      0.95      3322
weighted avg       0.98      0.98      0.98      3322



In [10]:
importance_df = pd.DataFrame({
    'Feature':    regression_features,
    'Importance': xgb_reg.feature_importances_
}).sort_values('Importance', ascending=False)

print('Top 15 features for demand forecasting:')
print(importance_df.head(15).to_string(index=False))

Top 15 features for demand forecasting:
      Feature  Importance
   unit_price    0.667777
demand_roll14    0.141935
demand_roll30    0.076904
 demand_roll7    0.014374
demand_roll90    0.011765
 category_enc    0.007478
   is_weekend    0.006943
  day_of_week    0.005810
 demand_std30    0.005394
        month    0.004958
 week_of_year    0.004733
 day_of_month    0.004590
  demand_lag7    0.004468
         year    0.003976
  demand_std7    0.003638


In [11]:
# Inventory optimizer: reorder recommendations for all drugs
drug_stats = df_feat.groupby('drug_name').agg(
    avg_daily_demand = ('demand_roll30', 'last'),
    std_daily_demand = ('demand_std30',  'last'),
    lead_time_days   = ('lead_time_days','first'),
    unit_price       = ('unit_price',    'first'),
    latest_stock     = ('stock_level',   'last'),
    reorder_point    = ('reorder_point', 'last'),
    safety_stock     = ('safety_stock',  'last'),
).reset_index()

recommendations = []
for _, row in drug_stats.iterrows():
    avg   = row['avg_daily_demand']
    stock = row['latest_stock']
    rop   = row['reorder_point']
    lead  = row['lead_time_days']
    opt30 = avg * 30 + row['safety_stock']
    days  = round(stock / avg, 1) if avg > 0 else 999

    if stock <= 0:
        status = 'OUT_OF_STOCK'
        action = f'URGENT: Order {int(opt30)} units immediately'
    elif stock <= rop:
        status = 'CRITICAL'
        action = f'Order {max(0,int(opt30-stock))} units NOW — {days} days left'
    elif stock <= rop * 1.5:
        status = 'LOW'
        action = f'Plan order of {max(0,int(opt30-stock))} units within {max(0,int(days-lead))} days'
    elif stock >= avg * 90:
        status = 'OVERSTOCK'
        action = f'Excess {int(stock-opt30)} units — wastage risk'
    else:
        status = 'NORMAL'
        action = 'No action needed'

    recommendations.append({
        'drug': row['drug_name'],
        'current_stock': int(stock),
        'avg_daily': round(avg,2),
        'days_left': days,
        'reorder_pt': round(rop,1),
        'optimal_30d': round(opt30,1),
        'status': status,
        'action': action
    })

rec_df = pd.DataFrame(recommendations).sort_values('days_left')
print('INVENTORY RECOMMENDATION REPORT')
print(rec_df.to_string(index=False))

INVENTORY RECOMMENDATION REPORT
 drug  current_stock  avg_daily  days_left  reorder_pt  optimal_30d status           action
N02BE            762      36.32       21.0       145.5       1126.1 NORMAL No action needed
M01AB            121       5.65       21.4        23.6        176.1 NORMAL No action needed
  R06             70       3.02       23.1        22.0         97.6 NORMAL No action needed
M01AE             95       4.07       23.4        17.7        127.5 NORMAL No action needed
 N05B            232       8.05       28.8        72.1        257.3 NORMAL No action needed
  R03            134       4.65       28.8        75.9        168.9 NORMAL No action needed
N02BA             94       3.20       29.4        13.7        100.0 NORMAL No action needed
 N05C             20       0.47       42.9         7.3         18.0 NORMAL No action needed


In [12]:
print('SHORTAGE ALERT (< 7 days of stock)')
shortage = rec_df[rec_df['days_left'] < 7]
print(shortage[['drug','current_stock','days_left','action']].to_string(index=False) if len(shortage) else 'None')

print('\nWASTAGE ALERT (Overstock)')
overstock = rec_df[rec_df['status'] == 'OVERSTOCK']
print(overstock[['drug','current_stock','optimal_30d','action']].to_string(index=False) if len(overstock) else 'None')

print('\nCRITICAL STATUS DRUGS')
critical = rec_df[rec_df['status'].isin(['CRITICAL','OUT_OF_STOCK'])]
print(critical.to_string(index=False) if len(critical) else 'None')

SHORTAGE ALERT (< 7 days of stock)
None

WASTAGE ALERT (Overstock)
None

CRITICAL STATUS DRUGS
None


In [13]:
def forecast_demand(drug_name, days_ahead=14):
    drug_df   = df_feat[df_feat['drug_name'] == drug_name].copy()
    last_row  = drug_df.iloc[-1].copy()
    last_date = drug_df['date'].iloc[-1]

    results = []
    for i in range(1, days_ahead + 1):
        future_date = last_date + pd.Timedelta(days=i)
        inp = last_row[regression_features].copy()
        inp['day_of_week']    = future_date.dayofweek
        inp['day_of_month']   = future_date.day
        inp['month']          = future_date.month
        inp['year']           = future_date.year
        inp['week_of_year']   = future_date.isocalendar()[1]
        inp['quarter']        = future_date.quarter
        inp['is_weekend']     = int(future_date.dayofweek >= 5)
        inp['is_month_start'] = int(future_date.day == 1)
        inp['season']         = {12:1,1:1,2:1,3:2,4:2,5:2,6:3,7:3,8:3,9:4,10:4,11:4}[future_date.month]

        pred = max(0, round(xgb_reg.predict(pd.DataFrame([inp.astype('float64')]))[0], 2))
        results.append({'date': str(future_date.date()), 'predicted_demand': pred})

    result_df = pd.DataFrame(results)
    total = result_df['predicted_demand'].sum()
    print(f'\n{drug_name}: {days_ahead}-day forecast')
    print(result_df.to_string(index=False))
    print(f'Total: {total:.1f} units | Suggested order (15% buffer): {round(total*1.15)} units')
    return result_df

# Forecast all drugs for next 14 days
for drug in drug_cols:
    forecast_demand(drug, days_ahead=14)


M01AB: 14-day forecast
      date  predicted_demand
2019-10-09              5.59
2019-10-10              6.27
2019-10-11              6.38
2019-10-12              6.54
2019-10-13              5.68
2019-10-14              5.38
2019-10-15              5.36
2019-10-16              5.24
2019-10-17              5.19
2019-10-18              5.28
2019-10-19              5.45
2019-10-20              5.54
2019-10-21              5.26
2019-10-22              5.26
Total: 78.4 units | Suggested order (15% buffer): 90 units

M01AE: 14-day forecast
      date  predicted_demand
2019-10-09              3.62
2019-10-10              4.29
2019-10-11              4.34
2019-10-12              4.51
2019-10-13              3.89
2019-10-14              3.56
2019-10-15              3.54
2019-10-16              3.50
2019-10-17              3.46
2019-10-18              3.50
2019-10-19              3.69
2019-10-20              3.81
2019-10-21              3.50
2019-10-22              3.50
Total: 52.7 units | Sug

In [14]:
def predict_for_drug(input_data: dict):
    """
    Predict stock status and tomorrow demand for a drug.

    Required: drug_name, date, current_stock
    Optional: demand_roll7, demand_roll30, demand_std30
              (auto-filled from training history if not provided)

    drug_name must be one of: M01AB, M01AE, N02BA, N02BE, N05B, N05C, R03, R06
    """
    drug  = input_data['drug_name']
    date  = pd.to_datetime(input_data['date'])
    stock = input_data['current_stock']

    hist   = df_feat[df_feat['drug_name'] == drug].iloc[-1]
    roll7  = input_data.get('demand_roll7',  hist['demand_roll7'])
    roll14 = hist['demand_roll14']
    roll30 = input_data.get('demand_roll30', hist['demand_roll30'])
    roll90 = hist['demand_roll90']
    std7   = hist['demand_std7']
    std30  = input_data.get('demand_std30',  hist['demand_std30'])
    lead   = drug_meta[drug]['lead_time_days']

    safety  = 1.5 * std30 * np.sqrt(lead)
    reorder = roll30 * lead + safety
    days_left = min(stock / roll7, 365) if roll7 > 0 else 999
    opt30 = roll30 * 30 + safety

    row_reg = {
        'drug_enc':        le_drug.transform([drug])[0],
        'category_enc':    hist['category_enc'],
        'day_of_week':     date.dayofweek,
        'day_of_month':    date.day,
        'month':           date.month,
        'year':            date.year,
        'week_of_year':    date.isocalendar()[1],
        'quarter':         date.quarter,
        'is_weekend':      int(date.dayofweek >= 5),
        'is_month_start':  int(date.day == 1),
        'is_month_end':    int(date.day == pd.Period(str(date)[:7]).days_in_month),
        'season':          {12:1,1:1,2:1,3:2,4:2,5:2,6:3,7:3,8:3,9:4,10:4,11:4}[date.month],
        'demand_lag1':     hist['demand_lag1'],
        'demand_lag7':     hist['demand_lag7'],
        'demand_lag14':    hist['demand_lag14'],
        'demand_lag30':    hist['demand_lag30'],
        'demand_roll7':    roll7,
        'demand_roll14':   roll14,
        'demand_roll30':   roll30,
        'demand_roll90':   roll90,
        'demand_std7':     std7,
        'demand_std30':    std30,
        'demand_trend':    roll7 - roll30,
        'stock_level':     stock,
        'stock_lag1':      hist['stock_level'],
        'lead_time_days':  lead,
        'unit_price':      drug_meta[drug]['unit_price'],
    }
    row_cls = {**row_reg, 'days_of_stock': days_left,
               'reorder_point': reorder, 'safety_stock': safety}

    pred_demand = max(0, round(
        xgb_reg.predict(pd.DataFrame([row_reg])[regression_features].astype('float64'))[0], 2
    ))
    status_enc  = xgb_clf.predict(pd.DataFrame([row_cls])[classification_features].astype('float64'))[0]
    status      = le_status.inverse_transform([status_enc])[0]
    proba       = xgb_clf.predict_proba(pd.DataFrame([row_cls])[classification_features].astype('float64'))[0]
    proba_dict  = dict(zip(le_status.classes_, proba.round(3)))
    order_qty   = max(0, int(opt30 - stock))

    print(f'\n{"="*55}')
    print(f'  Drug            : {drug} ({drug_meta[drug]["category"]})')
    print(f'  Date            : {input_data["date"]}')
    print(f'  Current stock   : {stock} units')
    print(f'  Days of stock   : {days_left:.1f} days')
    print(f'  Reorder point   : {reorder:.1f} units')
    print(f'  Stock status    : {status}')
    print(f'  Tomorrow demand : {pred_demand} units')
    if status in ['CRITICAL', 'OUT_OF_STOCK', 'LOW']:
        print(f'  Suggested order : {order_qty} units')
    elif status == 'OVERSTOCK':
        print(f'  Excess stock    : {int(stock - opt30)} units (wastage risk)')
    print(f'  Status proba    : {proba_dict}')
    print(f'{"="*55}')

    return {'status': status, 'predicted_demand': pred_demand,
            'days_of_stock': round(days_left,1), 'reorder_point': round(reorder,1),
            'suggested_order': order_qty, 'probabilities': proba_dict}


# Example predictions
predict_for_drug({'drug_name': 'N02BE', 'date': '2019-10-01', 'current_stock': 20})
predict_for_drug({'drug_name': 'M01AE', 'date': '2019-10-01', 'current_stock': 500})
predict_for_drug({'drug_name': 'N05B',  'date': '2019-10-01', 'current_stock': 2000})
predict_for_drug({'drug_name': 'R03',   'date': '2019-10-01', 'current_stock': 80})


  Drug            : N02BE (Painkiller)
  Date            : 2019-10-01
  Current stock   : 20 units
  Days of stock   : 0.6 days
  Reorder point   : 145.5 units
  Stock status    : CRITICAL
  Tomorrow demand : 42.540000915527344 units
  Suggested order : 1106 units
  Status proba    : {'CRITICAL': 0.839, 'LOW': 0.147, 'NORMAL': 0.015, 'OVERSTOCK': 0.0}

  Drug            : M01AE (Anti-inflammatory)
  Date            : 2019-10-01
  Current stock   : 500 units
  Days of stock   : 98.4 days
  Reorder point   : 17.7 units
  Stock status    : NORMAL
  Tomorrow demand : 4.840000152587891 units
  Status proba    : {'CRITICAL': 0.0, 'LOW': 0.0, 'NORMAL': 1.0, 'OVERSTOCK': 0.0}

  Drug            : N05B (Anxiolytic)
  Date            : 2019-10-01
  Current stock   : 2000 units
  Days of stock   : 212.1 days
  Reorder point   : 72.1 units
  Stock status    : NORMAL
  Tomorrow demand : 12.59000015258789 units
  Status proba    : {'CRITICAL': 0.0, 'LOW': 0.0, 'NORMAL': 1.0, 'OVERSTOCK': 0.0}

  Dr

{'status': 'LOW',
 'predicted_demand': 5.21,
 'days_of_stock': 16.0,
 'reorder_point': 75.9,
 'suggested_order': 88,
 'probabilities': {'CRITICAL': 0.069,
  'LOW': 0.853,
  'NORMAL': 0.078,
  'OVERSTOCK': 0.0}}

In [15]:
# Monthly consumption summary for procurement planning
monthly = df.copy()
monthly['month_year'] = monthly['date'].dt.to_period('M')

monthly_summary = monthly.groupby(['drug_name','month_year']).agg(
    total_dispensed = ('quantity_dispensed','sum'),
    avg_daily       = ('quantity_dispensed','mean'),
    max_daily       = ('quantity_dispensed','max'),
).reset_index()
monthly_summary = monthly_summary.round(2)

print('Monthly consumption — last 6 months per drug:')
print(monthly_summary.groupby('drug_name').tail(6).to_string(index=False))

Monthly consumption — last 6 months per drug:
drug_name month_year  total_dispensed  avg_daily  max_daily
    M01AB    2019-05           168.04       5.42      11.00
    M01AB    2019-06           151.54       5.05      11.34
    M01AB    2019-07           181.00       5.84      12.50
    M01AB    2019-08           181.91       5.87      13.84
    M01AB    2019-09           161.07       5.37      10.68
    M01AB    2019-10            44.37       5.55      11.34
    M01AE    2019-05            97.26       3.14       6.40
    M01AE    2019-06           101.63       3.39       6.33
    M01AE    2019-07           103.54       3.34       7.00
    M01AE    2019-08            88.27       2.85       6.85
    M01AE    2019-09           111.44       3.71       9.53
    M01AE    2019-10            37.30       4.66      11.69
    N02BA    2019-05           104.10       3.36       6.30
    N02BA    2019-06           103.20       3.44      10.50
    N02BA    2019-07            92.80       2.99      